# 01 Validation and Exploratory Data Analysis (EDA)

**Purpose:** Validate SQL ground truth metrics against Python-calculated aggregations to ensure strict analytical reconciliation. Once validated, establish a distributional understanding of catalog profitability, operational risk, rank divergence, and discount dependency before any quadrant modeling or scenario simulation begins.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

# plotting style for downstream EDA
plt.style.use('default')
sns.set_theme(style="whitegrid")

# Load environment variables
load_dotenv(dotenv_path='../src/config/.env')

DB_USER = os.getenv('DB_USER', 'postgres')
DB_PASSWORD = os.getenv('DB_PASSWORD', 'password')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'revenue_rank_reality')

# Establish SQLAlchemy connection
engine = create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

print("Environment setup complete. Database connection initialized.")

Environment setup complete. Database connection initialized.


In [2]:
# Load the primary analytical dataframe from the validated SQL export
csv_path = '../data/processed/ground_truth_master.csv'
df = pd.read_csv(csv_path)

print(f"Dataframe loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head(3)

Dataframe loaded successfully: 118 rows, 17 columns.


,product_card_id,product_name,category_name,total_revenue,total_profit,net_profit_pct,revenue_rank,profit_rank,rank_divergence,negative_profit_order_pct,discount_dependency_rate,total_orders,units_sold_per_day,late_delivery_risk_rate,avg_shipping_delay_days,profit_ratio_volatility,delivery_risk_rank
0,1004,Field & Stream Sportsman 16 Gun Fire Safe,Fishing,6637668.10,731576.18,0.1102,1,1,0,18.36,94.41,16595,14.74,0.5734,0.56,0.4670,43.0
1,365,Perfect Fitness Perfect Rip Deck,Cleats,4233794.25,473820.89,0.1119,2,2,0,18.66,94.41,23478,62.68,0.5739,0.58,0.4567,42.0
2,957,Diamondback Women's Serene Classic Comfort Bi,Camping & Hiking,3946836.86,409895.62,0.1039,3,3,0,18.89,94.41,13157,11.68,0.5691,0.55,0.4693,51.0


## 2. SQL Benchmark Reconciliation
All Python-calculated metrics must reconcile against the SQL ground truth within a ±0.5% tolerance. Any breach stops the pipeline.

In [4]:
# Ground truth targets from methodology_notes.md
sql_benchmarks = {
    'total_unique_products': 118,
    'total_revenue': 35214428.98,
    'avg_profit_ratio': 0.1208,
    'loss_making_products': 118,
    'avg_late_delivery_risk': 0.5729
}

# Re-calculate avg_profit_ratio directly from the fact table to match transactional grain
profit_ratio_query = "SELECT order_item_profit_ratio FROM analytics.fact_orders"
fact_profit_ratios = pd.read_sql(profit_ratio_query, engine)
py_avg_profit_ratio = round(fact_profit_ratios['order_item_profit_ratio'].mean(), 4)

# Python reconstruction from ground_truth_master.csv for the rest
py_metrics = {
    'total_unique_products': df['product_card_id'].nunique(),
    'total_revenue': round(df['total_revenue'].sum(), 2),
    'avg_profit_ratio': py_avg_profit_ratio, 
    'loss_making_products': df[df['negative_profit_order_pct'] > 0]['product_card_id'].nunique(),
    'avg_late_delivery_risk': round((df['late_delivery_risk_rate'] * df['total_orders']).sum() / df['total_orders'].sum(), 4) 
}

# Validation
print("--- RECONCILIATION AUDIT ---")
tolerance = 0.005 # 0.5%
failed = False

for metric, sql_val in sql_benchmarks.items():
    py_val = py_metrics[metric]
    if sql_val == 0:
        diff_pct = 0 if py_val == 0 else 1.0
    else:
        diff_pct = abs((py_val - sql_val) / sql_val)

    status = "PASS" if diff_pct <= tolerance else "FAIL"
    if status == "FAIL":
        failed = True
        
    print(f"{metric:<25} | SQL: {sql_val:<15} | PY: {py_val:<15} | DIFF: {diff_pct:.4%} | {status}")

if failed:
    raise ValueError("FATAL: Ground truth reconciliation failed. Investigate metric divergence before proceeding.")
else:
    print("\nSTATUS: ALL BENCHMARKS RECONCILED WITHIN TOLERANCE.")

--- RECONCILIATION AUDIT ---
total_unique_products     | SQL: 118             | PY: 118             | DIFF: 0.0000% | PASS
total_revenue             | SQL: 35214428.98     | PY: 35214428.98     | DIFF: 0.0000% | PASS
avg_profit_ratio          | SQL: 0.1208          | PY: 0.1208          | DIFF: 0.0000% | PASS
loss_making_products      | SQL: 118             | PY: 118             | DIFF: 0.0000% | PASS
avg_late_delivery_risk    | SQL: 0.5729          | PY: 0.5729          | DIFF: 0.0000% | PASS

STATUS: ALL BENCHMARKS RECONCILED WITHIN TOLERANCE.
